# 01: SmolVLAの推論

観測をSmolVLAに渡し、返ってきた行動をLift環境で実行します。依存ライブラリと学習済みモデルは実行時に取得します。

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `01_smolvla_inference` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `lift.json`
- `lift_tools.py`
- `prepare.py`
- `requirements.txt`
- `smolvla.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/01_smolvla_inference"  # @param {type:"string"}
REQUIRED_FILES = [
    "lift.json",
    "lift_tools.py",
    "prepare.py",
    "requirements.txt",
    "smolvla.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_01")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## SmolVLAは何を見て、何を返すか

00では、自分で決めた7個の値を `action` として送っていました。この回では、SmolVLAが次の3種類の情報から行動を予測します。

| 入力 | 何の情報か |
| --- | --- |
| 外側のカメラと手首のカメラの画像 | 作業台や立方体、手先の周りが今どう見えているか |
| 指示の文章 | 何をするか。初期設定は `pick up the cube`（立方体を持ち上げる） |
| 00で見た `state`（8個の値） | 手先の位置と向き、左右の指の関節位置 |

`select_action()` から受け取るのは **1ステップ分の行動**です。先頭の3個は手先の移動、次の3個は回転、最後の1個はグリッパーの開閉を表します。この7個を使って `env.step(action)` を実行すると、新しい画像と状態が得られます。この回はその新しい観測で、次の行動をまた予測します。

下のセルで、モデルを読み込む処理と、次の3段階で行動を予測する処理を用意します。

1. `pre(batch)` で指示文をトークンIDの列に変換し、`state` を正規化します。画像は画素の配列としてモデルに渡します。
2. `policy.select_action(model_inputs)` の中で、SmolVLAが画像を特徴に変換し、指示と状態も使って行動を生成します。Flow Matchingで学習した部分（Action Expert）が、ノイズから少しずつ行動の列を作ります。この教材ではその先頭の1ステップ分を使います。
3. `post(model_output)` で、予測した行動を学習時の平均・標準偏差を使って指令のスケールに戻します。

[LeRobotのSmolVLAドキュメント](https://huggingface.co/docs/lerobot/en/smolvla) · [SmolVLAの論文](https://arxiv.org/html/2506.01844v1)

In [ ]:
MODEL_CODE = r"""
import torch
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla import processor_smolvla  # noqa: F401

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

from smolvla import checkpoint_inputs, observation_batch


def load_model(settings, image_size=256):
    checkpoint, config, vlm_repo = checkpoint_inputs(settings, image_size)
    policy = SmolVLAPolicy.from_pretrained(checkpoint, config=config, strict=True).eval()
    # 正規化・トークン化・行動の逆正規化はチェックポイント付属のものを使います。
    pre, post = make_pre_post_processors(
        config,
        pretrained_path=checkpoint,
        preprocessor_overrides={
            "device_processor": {"device": config.device},
            "tokenizer_processor": {"tokenizer_name": config.vlm_model_name},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return policy, pre, post, vlm_repo


def predict_action(policy, pre, post, batch):
    with torch.inference_mode():
        # 指示文をトークンIDにし、stateを正規化します。
        model_inputs = pre(batch)
        # 画像・指示・stateを使い、モデルの内部で行動を生成します。
        model_output = policy.select_action(model_inputs)
        # 行動の値を、シミュレータに渡す指令のスケールへ戻します。
        predicted_action = post(model_output)
    return predicted_action.squeeze(0).detach().cpu().numpy()

"""

## 使うモデル、カメラ、指示を確認する

このセルでは、どの学習済みモデルを使うか、どのカメラの画像を渡すか、どんな指示で動かすかを表示します。実際に使った画像と状態は、推論を実行したあとのセルで見られます。

In [ ]:
settings = json.loads((PROJECT / "lift.json").read_text())
policy_config = settings["policy"]
print("モデル:", policy_config["path"])
print("固定revision:", policy_config["revision"])
print("カメラ:", policy_config["camera_map"])
print("指示:", settings["instruction"])

## SmolVLAを読み込み、行動を実行する

リセット直後は00と同じ固定指令で環境を落ち着かせます。その後は、次の流れを1ステップずつ繰り返します。

1. `obs` は今の観測です。カメラ画像とロボットの状態を取り出し、指示の文章と合わせて `batch` にまとめます。
2. `predicted_action` が **SmolVLAの予測した7個の値** です。手先の移動・回転と、グリッパーの開閉の指令が入っています。
3. `action` が **実際にシミュレータへ送る7個の値** です。予測値が許容範囲を外れていたら、`np.clip()` で上下限に収めます。
4. `env.step(action)` で1ステップ進めて、新しい `obs` を受け取ります。この観測を使って次の行動を予測します。

In [ ]:
result = run_python(
    MODEL_CODE
    + r"""
import json
import os
import tempfile
from pathlib import Path

import imageio.v2 as imageio
import numpy as np

# 描画バックエンドをrobosuiteのimportより先に設定します。
settings = json.loads(Path("lift.json").read_text())
config = settings["env"]
os.environ["MUJOCO_GL"] = config["renderer"]
os.environ["PYOPENGL_PLATFORM"] = config["renderer"]
import robosuite
from robosuite import macros
from lift_tools import lift_controller

macros.IMAGE_CONVENTION = "opengl"
torch.manual_seed(settings["seed"])
policy, pre, post, _ = load_model(settings["policy"], config["image_size"])
env = robosuite.make(
    env_name=config["task"],
    robots=config["robot"],
    controller_configs=lift_controller(),
    has_renderer=False,
    has_offscreen_renderer=True,
    use_camera_obs=True,
    use_object_obs=False,
    camera_names=config["cameras"],
    camera_heights=config["image_size"],
    camera_widths=config["image_size"],
    control_freq=config["control_freq"],
    horizon=settings["max_steps"] + config["settle_steps"],
    hard_reset=False,
    reward_shaping=False,
    seed=settings["seed"],
)
try:
    obs = env.reset()
    # 00と同じ固定指令で、初期配置が落ち着くまで進めます。
    for _ in range(config["settle_steps"]):
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])

    folder = Path(tempfile.mkdtemp(prefix="lift-", dir="outputs")).resolve()
    camera_key = f'{config["cameras"][0]}_image'
    camera_map = settings["policy"]["camera_map"]
    steps = []
    with imageio.get_writer(folder / "video.mp4", fps=config["control_freq"]) as video:
        video.append_data(obs[camera_key][::-1])
        for step in range(settings["max_steps"]):
            # batchは今の画像・指示・stateをまとめた入力です。
            batch = observation_batch(obs, settings["instruction"], settings["policy"])
            state_before = batch["observation.state"].tolist()
            if step == 0:
                # モデルに渡す画像を、表示用にRGB画像に戻して並べます。
                input_views = [
                    np.rint(batch[key].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
                    for key in camera_map
                ]
                imageio.imwrite(folder / "model_inputs.png", np.concatenate(input_views, axis=1))

            # predicted_actionが、SmolVLAの予測した7個の値です。
            predicted_action = predict_action(policy, pre, post, batch)
            # actionが実際に送る指令です。env.action_specはその上下限です。
            action = np.clip(predicted_action, *env.action_spec)
            # 返ってきたobsを、次のステップの入力に使います。
            obs, reward, done, _ = env.step(action)
            video.append_data(obs[camera_key][::-1])
            steps.append(
                {
                    "step": step,
                    "state_before": state_before,
                    "predicted_action": predicted_action.tolist(),
                    "action": action.tolist(),
                    "reward": reward,
                }
            )
            if done or reward > 0:
                break
    report = {
        "camera_shape": list(obs[camera_key].shape),
        "input_cameras": list(camera_map.values()),
        "state_order": ["x", "y", "z", "rx", "ry", "rz", "gripper_left", "gripper_right"],
        "action_order": ["dx", "dy", "dz", "drx", "dry", "drz", "gripper"],
        "steps": steps,
    }
    report["model"] = settings["policy"]["path"]
    report["instruction"] = settings["instruction"]
    (folder / "observations.json").write_text(json.dumps(report, indent=2))
    print(f"Results: {folder}")
finally:
    env.close()
"""
)

## 1回目の推論で使った入力と、返ってきた行動を見る

最初の1ステップを取り出して表示します。画像はモデルに渡したときの向きです。初期設定では左が外側、右が手首のカメラです。出力の表では「モデルの予測」が `predicted_action`、「実際に送った指令」が `action` です。下の動画で、その後も新しい観測を使って予測と行動を繰り返す様子を見てください。

In [ ]:
from IPython.display import Image, Markdown, Video, display

report = json.loads((result / "observations.json").read_text())
first = report["steps"][0]
print("モデル:", report["model"])

display(Markdown("### 入力"))
print("言語指示:", report["instruction"])
print("カメラ画像（左から）:", " / ".join(report["input_cameras"]))
display(Image(filename=str(result / "model_inputs.png")))
state_rows = ["| stateの項目 | 値 |", "| --- | ---: |"]
for name, value in zip(report["state_order"], first["state_before"], strict=True):
    state_rows.append(f"| `{name}` | {value:+.5f} |")
display(Markdown("\n".join(state_rows)))

display(Markdown("### 出力"))
action_rows = ["| actionの項目 | モデルの予測 | 実際に送った指令 |", "| --- | ---: | ---: |"]
for name, predicted, executed in zip(
    report["action_order"], first["predicted_action"], first["action"], strict=True
):
    action_rows.append(f"| `{name}` | {predicted:+.5f} | {executed:+.5f} |")
display(Markdown("\n".join(action_rows)))

display(Markdown("### 行動を繰り返した結果"))
display(Video(str(result / "video.mp4"), embed=True))

## 確認課題

2枚の入力画像には何が映っていますか。出力の `gripper` を見て、最初にどんな開閉指令を送ったかを00の例と照らし合わせてください。動画で手先がその後どう動くかも確認しましょう。

## 結果をDriveに保存する

最初の推論の入力画像、推論時の動画、実行設定、状態と行動のログを保存します。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = (
    "01_smolvla_inference_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
)
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)